In [ ]:
!pip install catboost

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Step 1: Load the Base Models (DenseNet169, Xception, and EfficientNetB4 for Fruit)
fruit_model_1_path = '/content/drive/MyDrive/Capstone Models/DenseNet169_fruit.keras'
fruit_model_2_path = '/content/drive/MyDrive/Capstone Models/Xception_fruit.keras'
fruit_model_3_path = '/content/drive/MyDrive/Capstone Models/EfficientNetB4_fruit.keras'

fruit_model_1 = load_model(fruit_model_1_path)
fruit_model_2 = load_model(fruit_model_2_path)
fruit_model_3 = load_model(fruit_model_3_path)

In [ ]:
# Step 2: Load the Dataset (Fruit Test Dataset)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

test_dir = "/content/drive/MyDrive/Capstone Dataset/Guava_Fruit_Dieases/Test"

# Define fruit classes manually
fruit_classes = ['Anthracnose', 'Healthy', 'Scab', 'Styler end root']

# Load the test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 880 files belonging to 4 classes.


In [ ]:
# Step 3: Generate Predictions from Base Models
y_prob_fruit_model_1 = fruit_model_1.predict(test_ds)  # Predictions from DenseNet169
y_prob_fruit_model_2 = fruit_model_2.predict(test_ds)  # Predictions from Xception
y_prob_fruit_model_3 = fruit_model_3.predict(test_ds)  # Predictions from EfficientNetB4

# Step 4: Stack the Predictions (Create Meta-Features)
X_meta_fruit = np.column_stack((
    y_prob_fruit_model_1,
    y_prob_fruit_model_2,
    y_prob_fruit_model_3
))

# True labels for the test set (adjust this to match your dataset)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

28/28 ━━━━━━━━━━━━━━━━━━━━ 211s 7s/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 10s 229ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 340ms/step


# Step 5: Define and Train Meta-Models (Each Section)

In [ ]:
## 1. Logistic Regression
print("\nTraining Logistic Regression...")
meta_model_lr = LogisticRegression()
meta_model_lr.fit(X_meta_fruit, y_true)
y_meta_pred_lr = meta_model_lr.predict(X_meta_fruit)
print(f"Logistic Regression Accuracy: {accuracy_score(y_true, y_meta_pred_lr):.4f}")
print(classification_report(y_true, y_meta_pred_lr, target_names=fruit_classes))


Training Logistic Regression...
Logistic Regression Accuracy: 0.7830
                 precision    recall  f1-score   support

    Anthracnose       0.75      0.55      0.63       220
        Healthy       0.78      0.94      0.85       220
           Scab       0.75      0.76      0.76       220
Styler end root       0.85      0.89      0.87       220

       accuracy                           0.78       880
      macro avg       0.78      0.78      0.78       880
   weighted avg       0.78      0.78      0.78       880



In [ ]:
## 2. Random Forest
print("\nTraining Random Forest...")
meta_model_rf = RandomForestClassifier()
meta_model_rf.fit(X_meta_fruit, y_true)
y_meta_pred_rf = meta_model_rf.predict(X_meta_fruit)
print(f"Random Forest Accuracy: {accuracy_score(y_true, y_meta_pred_rf):.4f}")
print(classification_report(y_true, y_meta_pred_rf, target_names=fruit_classes))


Training Random Forest...
Random Forest Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 3. Gradient Boosting
print("\nTraining Gradient Boosting...")
meta_model_gb = GradientBoostingClassifier()
meta_model_gb.fit(X_meta_fruit, y_true)
y_meta_pred_gb = meta_model_gb.predict(X_meta_fruit)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_true, y_meta_pred_gb):.4f}")
print(classification_report(y_true, y_meta_pred_gb, target_names=fruit_classes))


Training Gradient Boosting...
Gradient Boosting Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 4. Support Vector Classifier (SVC)
print("\nTraining Support Vector Classifier (SVC)...")
meta_model_svc = SVC()
meta_model_svc.fit(X_meta_fruit, y_true)
y_meta_pred_svc = meta_model_svc.predict(X_meta_fruit)
print(f"SVC Accuracy: {accuracy_score(y_true, y_meta_pred_svc):.4f}")
print(classification_report(y_true, y_meta_pred_svc, target_names=fruit_classes))


Training Support Vector Classifier (SVC)...
SVC Accuracy: 0.8489
                 precision    recall  f1-score   support

    Anthracnose       0.87      0.74      0.80       220
        Healthy       0.79      0.97      0.87       220
           Scab       0.83      0.80      0.82       220
Styler end root       0.92      0.89      0.90       220

       accuracy                           0.85       880
      macro avg       0.85      0.85      0.85       880
   weighted avg       0.85      0.85      0.85       880



In [ ]:
## 5. Multi-layer Perceptron (MLP)
print("\nTraining Multi-layer Perceptron (MLP)...")
meta_model_mlp = MLPClassifier()
meta_model_mlp.fit(X_meta_fruit, y_true)
y_meta_pred_mlp = meta_model_mlp.predict(X_meta_fruit)
print(f"MLP Accuracy: {accuracy_score(y_true, y_meta_pred_mlp):.4f}")
print(classification_report(y_true, y_meta_pred_mlp, target_names=fruit_classes))


Training Multi-layer Perceptron (MLP)...
MLP Accuracy: 0.8636
                 precision    recall  f1-score   support

    Anthracnose       0.86      0.81      0.83       220
        Healthy       0.83      0.95      0.88       220
           Scab       0.87      0.81      0.84       220
Styler end root       0.91      0.89      0.90       220

       accuracy                           0.86       880
      macro avg       0.87      0.86      0.86       880
   weighted avg       0.87      0.86      0.86       880



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
## 6. AdaBoost Classifier
print("\nTraining AdaBoost Classifier...")
meta_model_ada = AdaBoostClassifier()
meta_model_ada.fit(X_meta_fruit, y_true)
y_meta_pred_ada = meta_model_ada.predict(X_meta_fruit)
print(f"AdaBoost Accuracy: {accuracy_score(y_true, y_meta_pred_ada):.4f}")
print(classification_report(y_true, y_meta_pred_ada, target_names=fruit_classes))


Training AdaBoost Classifier...
AdaBoost Accuracy: 0.8205
                 precision    recall  f1-score   support

    Anthracnose       0.85      0.70      0.77       220
        Healthy       0.82      0.85      0.84       220
           Scab       0.78      0.83      0.80       220
Styler end root       0.84      0.89      0.86       220

       accuracy                           0.82       880
      macro avg       0.82      0.82      0.82       880
   weighted avg       0.82      0.82      0.82       880



In [ ]:
## 7. Decision Tree Classifier
print("\nTraining Decision Tree Classifier...")
meta_model_dt = DecisionTreeClassifier()
meta_model_dt.fit(X_meta_fruit, y_true)
y_meta_pred_dt = meta_model_dt.predict(X_meta_fruit)
print(f"Decision Tree Accuracy: {accuracy_score(y_true, y_meta_pred_dt):.4f}")
print(classification_report(y_true, y_meta_pred_dt, target_names=fruit_classes))


Training Decision Tree Classifier...
Decision Tree Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 8. K-Nearest Neighbors (KNN)
print("\nTraining K-Nearest Neighbors (KNN)...")
meta_model_knn = KNeighborsClassifier()
meta_model_knn.fit(X_meta_fruit, y_true)
y_meta_pred_knn = meta_model_knn.predict(X_meta_fruit)
print(f"KNN Accuracy: {accuracy_score(y_true, y_meta_pred_knn):.4f}")
print(classification_report(y_true, y_meta_pred_knn, target_names=fruit_classes))


Training K-Nearest Neighbors (KNN)...
KNN Accuracy: 0.9409
                 precision    recall  f1-score   support

    Anthracnose       0.94      0.91      0.93       220
        Healthy       0.94      0.98      0.96       220
           Scab       0.93      0.91      0.92       220
Styler end root       0.95      0.96      0.95       220

       accuracy                           0.94       880
      macro avg       0.94      0.94      0.94       880
   weighted avg       0.94      0.94      0.94       880



In [ ]:
## 9. Gaussian Naive Bayes (GNB)
print("\nTraining Gaussian Naive Bayes (GNB)...")
meta_model_gnb = GaussianNB()
meta_model_gnb.fit(X_meta_fruit, y_true)
y_meta_pred_gnb = meta_model_gnb.predict(X_meta_fruit)
print(f"GNB Accuracy: {accuracy_score(y_true, y_meta_pred_gnb):.4f}")
print(classification_report(y_true, y_meta_pred_gnb, target_names=fruit_classes))


Training Gaussian Naive Bayes (GNB)...
GNB Accuracy: 0.7443
                 precision    recall  f1-score   support

    Anthracnose       0.69      0.43      0.53       220
        Healthy       0.75      0.92      0.83       220
           Scab       0.74      0.70      0.72       220
Styler end root       0.76      0.93      0.84       220

       accuracy                           0.74       880
      macro avg       0.74      0.74      0.73       880
   weighted avg       0.74      0.74      0.73       880



In [ ]:
## 10. Quadratic Discriminant Analysis (QDA)
print("\nTraining Quadratic Discriminant Analysis (QDA)...")
meta_model_qda = QuadraticDiscriminantAnalysis()
meta_model_qda.fit(X_meta_fruit, y_true)
y_meta_pred_qda = meta_model_qda.predict(X_meta_fruit)
print(f"QDA Accuracy: {accuracy_score(y_true, y_meta_pred_qda):.4f}")
print(classification_report(y_true, y_meta_pred_qda, target_names=fruit_classes))


Training Quadratic Discriminant Analysis (QDA)...
QDA Accuracy: 0.7795
                 precision    recall  f1-score   support

    Anthracnose       0.82      0.46      0.59       220
        Healthy       0.83      0.93      0.88       220
           Scab       0.72      0.79      0.75       220
Styler end root       0.77      0.95      0.85       220

       accuracy                           0.78       880
      macro avg       0.78      0.78      0.77       880
   weighted avg       0.78      0.78      0.77       880



/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


In [ ]:
## 11. XGBoost Classifier
print("\nTraining XGBoost Classifier...")
meta_model_xgb = XGBClassifier()
meta_model_xgb.fit(X_meta_fruit, y_true)
y_meta_pred_xgb = meta_model_xgb.predict(X_meta_fruit)
print(f"XGBoost Accuracy: {accuracy_score(y_true, y_meta_pred_xgb):.4f}")
print(classification_report(y_true, y_meta_pred_xgb, target_names=fruit_classes))


Training XGBoost Classifier...
XGBoost Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 12. CatBoost Classifier
print("\nTraining CatBoost Classifier...")
meta_model_cat = CatBoostClassifier(learning_rate=0.1, iterations=1000, depth=6)
meta_model_cat.fit(X_meta_fruit, y_true)
y_meta_pred_cat = meta_model_cat.predict(X_meta_fruit)
print(f"CatBoost Accuracy: {accuracy_score(y_true, y_meta_pred_cat):.4f}")
print(classification_report(y_true, y_meta_pred_cat, target_names=fruit_classes))


Training CatBoost Classifier...
0:	learn: 1.2608701	total: 57.5ms	remaining: 57.5s
1:	learn: 1.1689057	total: 65.2ms	remaining: 32.6s
2:	learn: 1.0812853	total: 72.7ms	remaining: 24.2s
3:	learn: 0.9997540	total: 80.2ms	remaining: 20s
4:	learn: 0.9351099	total: 88.5ms	remaining: 17.6s
5:	learn: 0.8793613	total: 96.1ms	remaining: 15.9s
6:	learn: 0.8249857	total: 104ms	remaining: 14.7s
7:	learn: 0.7798964	total: 112ms	remaining: 13.9s
8:	learn: 0.7390199	total: 120ms	remaining: 13.2s
9:	learn: 0.7033105	total: 128ms	remaining: 12.6s
10:	learn: 0.6703259	total: 140ms	remaining: 12.6s
11:	learn: 0.6397924	total: 151ms	remaining: 12.4s
12:	learn: 0.6107115	total: 166ms	remaining: 12.6s
13:	learn: 0.5838134	total: 176ms	remaining: 12.4s
14:	learn: 0.5578868	total: 184ms	remaining: 12.1s
15:	learn: 0.5373266	total: 193ms	remaining: 11.9s
16:	learn: 0.5187670	total: 201ms	remaining: 11.6s
17:	learn: 0.5009207	total: 208ms	remaining: 11.4s
18:	learn: 0.4829073	total: 217ms	remaining: 11.2s
19:	

In [ ]:
## 13. LightGBM Classifier
print("\nTraining LightGBM Classifier...")
meta_model_lgbm = LGBMClassifier()
meta_model_lgbm.fit(X_meta_fruit, y_true)
y_meta_pred_lgbm = meta_model_lgbm.predict(X_meta_fruit)
print(f"LightGBM Accuracy: {accuracy_score(y_true, y_meta_pred_lgbm):.4f}")
print(classification_report(y_true, y_meta_pred_lgbm, target_names=fruit_classes))


Training LightGBM Classifier...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000232 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 880, number of used features: 12
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
